# MAE pretraining (thin wrapper)
All logic lives in `scripts/`. Settings: **Accelerator = GPU T4**, **Internet = On** (CIFAR-10 download).
Run with **Save Version → Save & Run All (Commit)** so it keeps going with the browser closed.
Outputs (`/kaggle/working/checkpoints`, `/kaggle/working/results`) become this notebook's output; attach that
output as an input to the next session.

| Session | `BLOCKS` |
| --- | --- |
| Day 1a (interactive is fine, ~20 min) | `test timing lrcheck` |
| Day 1b (background) | `pretrain_m75` (set `LR_PT` if lrcheck picked 1e-3) |
| Day 2 (background) | `pretrain_abl` (optionally add `pretrain_m75e200`) |

If a session stops early, attach its output and rerun the same block: it resumes from `ckpt_latest.pt`.

In [ ]:
BLOCKS = "pretrain_m75"
LR_PT = ""   # "1e-3" only if the lrcheck block says so

In [ ]:
# --- Code: attach this repo as a Kaggle Dataset named "mae-code" (zip of the repo), OR set REPO_URL.
import glob, os, shutil, subprocess
REPO_URL = ""   # e.g. "https://github.com/<you>/mae-label-efficiency.git"
CODE = "/tmp/mae"   # outside /kaggle/working so code + smoke files are not saved as notebook output
shutil.rmtree(CODE, ignore_errors=True)
hits = glob.glob("/kaggle/input/**/scripts/run_block.sh", recursive=True)
if hits:                                   # repo attached as a dataset (any folder depth)
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
else:
    raise SystemExit("Attach the code dataset or set REPO_URL")
os.chdir(CODE)
os.environ.update(DATA="/kaggle/working/data", RESULTS="/kaggle/working/results",
                  CKPT="/kaggle/working/checkpoints",
                  SEARCH="/kaggle/working/checkpoints /kaggle/input")
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())"

In [ ]:
if LR_PT: os.environ["LR_PT"] = LR_PT
!bash scripts/run_block.sh {BLOCKS}

In [ ]:
!ls -la /kaggle/working/checkpoints/* /kaggle/working/results/figures 2>/dev/null | head -50
!tail -3 /kaggle/working/results/logs/pt_*.csv 2>/dev/null